In [60]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

True

In [61]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000020BCFB12AD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020BCFB134D0>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [62]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [63]:
config={"configurable":{"session_id":"chat1"}}

In [64]:
from langchain_core.messages import HumanMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Ujjwal and I am a Senior AI/ML Engineer")],
    config=config
)

In [65]:
print(response.content.split("</think>")[-1])



Hi Ujjwal, great to meet you! 👋 It's always a pleasure to connect with a Senior AI/ML Engineer. Whether you're diving into LLM fine-tuning, MLOps pipelines, model optimization, architecture design, or navigating the latest research and tooling, I'm here to help you think through problems, review code, or brainstorm solutions.

What are you currently working on or looking to tackle?


In [66]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
print(response.content.split("</think>")[-1])




Your name is Ujjwal. How can I assist you today?


In [67]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1,
)
print(response.content.split("</think>")[-1])




I don't actually know your name! I don't have access to personal information or memory of past conversations, so I only know what you share in our current chat. If you'd like to tell me your name (or a nickname), I'd be happy to use it. Otherwise, you're always welcome to just chat as you are! 😊


In [68]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
print(response.content.split("</think>")[-1])



Nice to meet you, John! I'll make sure to use your name in our conversation. How can I help you today? 😊


In [69]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
print(response.content.split("</think>")[-1])



Your name is John! How can I help you today? 😊


### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [70]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')] | typing.Annotated[langchain_core.messag

In [71]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Ujjwal")]})

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Ujjwal"\n   - This is a simple greeting and introduction.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name: "Ujjwal"\n   - Intent: Introduce themselves, likely expecting a friendly response.\n\n3.  **Determine Response Goals:**\n   - Acknowledge the greeting\n   - Use their name to personalize the response\n   - Maintain a friendly, helpful tone\n   - Invite them to share how I can assist\n\n4.  **Draft Response (Mental):**\n   Hi Ujjwal! Nice to meet you. How can I help you today?\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional\n   - Check personalization: Uses name correctly\n   - Check openness: Invites next question/request\n   - Matches guidelines: Concise, helpful\n\n   Final: "Hi Ujjwal! Nice to meet you. How can I assist you today?" \n\n6.  **Self-Correction/Verification:**\n   - No complex requirements\n   - Simple greet

In [72]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [73]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Krish"\n   - This is a simple greeting and introduction.\n   - No specific question or request is made.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name: "Krish"\n   - Implicit need: Acknowledge the greeting, use the name, and offer assistance.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting warmly\n   - Use the user\'s name (Krish)\n   - Offer help or ask how I can assist\n   - Keep it friendly and concise\n\n4.  **Draft Response (Mental):**\n   Hi Krish! Nice to meet you. How can I help you today?\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional\n   - Check accuracy: Uses correct name\n   - Check completeness: Invites next step\n   - All good.\n\n6.  **Final Output Generation:** (matches the refined draft)\n   "Hi Krish! Nice to meet you. How can I assist you today?"✅\n</think>\n\nHi Krish! Nice to meet yo

In [74]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

print(response.content.split("</think>")[-1])



Your name is Krish! How can I help you today?


In [75]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [76]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
print(response.content.split("</think>")[-1])



नमस्ते कृष्! आपका स्वागत है। बताइए, मैं आपकी कैसे मदद कर सकता हूँ? 😊


Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [77]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [78]:
config = {"configurable": {"session_id": "chat5"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Krish"
   - Language: English
   - Intent: Introduction/Greeting
   - Key information: Name is "Krish"

2.  **Identify Constraints:**
   - System prompt: "Answer all questions to the best of your ability in Hindi."
   - This means I must respond in Hindi, regardless of the input language.

3.  **Formulate Response (Mental Draft in Hindi):**
   - Acknowledge the greeting and name.
   - Respond warmly in Hindi.
   - Example: "नमस्ते कृष्ण! आपका स्वागत है। मैं आपकी कैसे मदद कर सकता हूँ?" (Hello Krish! Welcome. How can I help you?)

4.  **Check Against Constraints:**
   - Is it in Hindi? Yes.
   - Does it address the user's input? Yes, acknowledges the name and greeting.
   - Is it polite and helpful? Yes.

5.  **Refine Response (Hindi):**
   "नमस्ते कृष्ण! आपका स्वागत है। बताइए, मैं आपकी कैसे मदद कर सकता हूँ?"
   (Note: "Krish" is often a short form of "Krishna" or "Krishnakumar", but I'll stic

In [79]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [80]:
response.content.split("</think>")[-1]

'\n\nआपका नाम क्रिश है। बताइए, मैं आपकी और कैसे मदद कर सकता हूँ?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [81]:
from langchain_core.messages import SystemMessage, AIMessage, trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [82]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content.split("</think>")[-1]

"\n\nI don't actually know! I don't have access to your personal preferences or memory of past conversations, so I can't recall what ice cream you like. But if you tell me your favorite flavors, textures, or ingredients, I'd be happy to suggest some perfect scoops or help you pick your next treat! 🍦 What are you usually in the mood for?"

In [83]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content.split("</think>")[-1]

'\n\nYou asked "2 + 2".'

In [84]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [85]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content.split("</think>")[-1]

"\n\nI don't know your name yet—you haven't mentioned it! What should I call you?"

In [86]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content.split("</think>")[-1]

"\n\nThis is actually the first message you've sent in this conversation, so I don't have a record of any previous math problem. If you'd like, please paste or describe the problem here and I'll gladly help you work through it!"